In [1]:
import torch
import math

def rope(x, positions, base=10000.0):
    """
    x: (batch, seq_len, num_heads, head_dim) 
    positions: (seq_len,) 整数位置索引
    """
    batch, seq_len, num_heads, head_dim = x.shape
    d = head_dim
    
    # 1. 计算每对维度的 theta_i
    # 下标 i 从 0 到 d/2 -1
    i = torch.arange(0, d // 2, dtype=torch.float32, device=x.device)
    theta = 1.0 / (base ** (2 * i / d))   # shape: (d/2,)
    
    # 2. 计算每个位置的角度 m * theta
    # positions: (seq_len,) -> (seq_len, 1)
    # theta: (d/2,)  -> (1, d/2)
    m_theta = positions[:, None].float() * theta[None, :]   # (seq_len, d/2)
    
    # 3. 计算 cos 和 sin，扩维以适应 head_dim
    cos = torch.cos(m_theta)   # (seq_len, d/2)
    sin = torch.sin(m_theta)
    
    # 每个值要作用于相邻的两个元素，所以 repeat_interleave
    cos = torch.repeat_interleave(cos, 2, dim=-1)   # (seq_len, d)
    sin = torch.repeat_interleave(sin, 2, dim=-1)
    
    # 4. 重新排列 x 得到用于旋转的 -x1, x0 形式
    # x 的最后一维两两分组: [x0, x1, x2, x3, ...]
    # 需要变为 [-x1, x0, -x3, x2, ...]
    x_rot = torch.empty_like(x)
    x_rot[..., 0::2] = -x[..., 1::2]   # 偶数位置放负的奇数元素
    x_rot[..., 1::2] =  x[..., 0::2]   # 奇数位置放偶数元素
    
    # 5. 应用旋转：x * cos + x_rot * sin
    # 需要把 cos 和 sin 的维度对齐到 (batch, seq_len, num_heads, head_dim)
    cos = cos[None, :, None, :]   # (1, seq_len, 1, d)
    sin = sin[None, :, None, :]
    
    return x * cos + x_rot * sin

In [2]:
# 假设 2个头，维度4，长度3
x = torch.randn(1, 3, 2, 4)
pos = torch.arange(3)
out = rope(x, pos)
print(out.shape)  # 应该还是 (1,3,2,4)

torch.Size([1, 3, 2, 4])


In [ ]:
# i just thought this will be a big [[cosθi, sinθi], [-sinθi, cosθi]] * d // 2 to @ x

In [4]:
import torch
import math

def rope_with_print(x, positions, base=10000.0):
    """
    x: (batch, seq_len, num_heads, head_dim)
    positions: (seq_len,)
    """
    batch, seq_len, num_heads, head_dim = x.shape
    d = head_dim
    print(f"1. 输入 x 形状: {x.shape} = (batch={batch}, seq_len={seq_len}, heads={num_heads}, dim={d})")
    print("   x 值 (第一个样本, 第一个头, 所有位置):")
    print(x[0, :, 0, :])
    
    # === 计算 theta ===
    i = torch.arange(0, d // 2, dtype=torch.float32, device=x.device)
    print(f"\n2. i 索引: {i}  形状: {i.shape}  # 表示第 i 对维度")
    theta = 1.0 / (base ** (2 * i / d))
    print(f"   theta: {theta}  形状: {theta.shape}  # 每个维对的基础角度")
    
    # === 计算每个位置的 m*theta 矩阵 ===
    print(f"\n3. positions 形状: {positions.shape} = (seq_len,)")
    print(f"   positions[:, None] 形状: {positions[:, None].shape}  # 变成列向量")
    print(f"   theta[None, :] 形状: {theta[None, :].shape}      # 变成行向量")
    m_theta = positions[:, None].float() * theta[None, :]   # 外积广播
    print(f"   m_theta 形状: {m_theta.shape}  # (seq_len, d/2)")
    print("   m_theta 值:\n", m_theta)
    
    # === 计算 cos 和 sin ===
    cos = torch.cos(m_theta)   # (seq_len, d/2)
    sin = torch.sin(m_theta)
    print(f"\n4. cos 形状: {cos.shape}  # (seq_len, d/2)")
    print("   cos 值:\n", cos)
    print(f"   sin 形状: {sin.shape}")
    print("   sin 值:\n", sin)
    
    # === 将 cos/sin 的每个元素重复两次，对应 head_dim ===
    cos_r = torch.repeat_interleave(cos, 2, dim=-1)  # (seq_len, d)
    sin_r = torch.repeat_interleave(sin, 2, dim=-1)
    print(f"\n5. 用 repeat_interleave 把每个角度重复两次：")
    print(f"   cos 重复后形状: {cos_r.shape}  # (seq_len, d)")
    print("   cos 重复后值:\n", cos_r)
    
    # === 构造用于旋转的向量 x_rot ===
    print(f"\n6. 构造 x_rot (把每对 [x0, x1] 变成 [-x1, x0])")
    x_rot = torch.empty_like(x)
    x_rot[..., 0::2] = -x[..., 1::2]   # 偶数位置放负的奇数元素
    x_rot[..., 1::2] =  x[..., 0::2]   # 奇数位置放偶数元素
    print("   原始 x (第一个样本，第一个头，所有位置):")
    print(x[0, :, 0, :])
    print("   x_rot:")
    print(x_rot[0, :, 0, :])
    
    # === 对齐维度并应用旋转 ===
    # 需要把 cos_r 和 sin_r 从 (seq_len, d) 扩展到 (1, seq_len, 1, d)
    cos_r = cos_r[None, :, None, :]   # (1, seq_len, 1, d)
    sin_r = sin_r[None, :, None, :]
    print(f"\n7. cos 广播为 (1, seq_len, 1, d): {cos_r.shape}")
    print("   cos 值 (针对每个位置和维度):\n", cos_r[0, :, 0, :])
    
    out = x * cos_r + x_rot * sin_r
    print(f"\n8. 输出形状: {out.shape}")
    print("   输出 x (第一个样本，第一个头):")
    print(out[0, :, 0, :])
    
    return out

# 固定种子，方便查看数值
torch.manual_seed(42)
# 构造一个小例子: batch=1, seq_len=3, num_heads=1, head_dim=4
x = torch.randn(1, 3, 1, 4)
positions = torch.arange(x.shape[-3])   # 位置 0, 1, 2

rope_with_print(x, positions, base=10000.0)

1. 输入 x 形状: torch.Size([1, 3, 1, 4]) = (batch=1, seq_len=3, heads=1, dim=4)
   x 值 (第一个样本, 第一个头, 所有位置):
tensor([[ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349,  0.8094]])

2. i 索引: tensor([0., 1.])  形状: torch.Size([2])  # 表示第 i 对维度
   theta: tensor([1.0000, 0.0100])  形状: torch.Size([2])  # 每个维对的基础角度

3. positions 形状: torch.Size([3]) = (seq_len,)
   positions[:, None] 形状: torch.Size([3, 1])  # 变成列向量
   theta[None, :] 形状: torch.Size([1, 2])      # 变成行向量
   m_theta 形状: torch.Size([3, 2])  # (seq_len, d/2)
   m_theta 值:
 tensor([[0.0000, 0.0000],
        [1.0000, 0.0100],
        [2.0000, 0.0200]])

4. cos 形状: torch.Size([3, 2])  # (seq_len, d/2)
   cos 值:
 tensor([[ 1.0000,  1.0000],
        [ 0.5403,  0.9999],
        [-0.4161,  0.9998]])
   sin 形状: torch.Size([3, 2])
   sin 值:
 tensor([[0.0000, 0.0000],
        [0.8415, 0.0100],
        [0.9093, 0.0200]])

5. 用 repeat_interleave 把每个角度重复两次：
   cos 重复后形状: torch.Size([

tensor([[[[ 0.3367,  0.1288,  0.2345,  0.2303]],

         [[-0.4499, -1.0455,  2.2145, -0.6159]],

         [[-0.4352,  0.3085,  0.5186,  0.8199]]]])

In [5]:
import torch
import math

def precompute_rope_cos_sin(dim, seq_len, base=10000.0, device='cpu'):
    """
    预计算 RoPE 所需的 cos 和 sin 表
    返回: cos, sin  形状均为 (seq_len, dim)
    """
    # dim 必须是偶数
    # 频率计算：1 / (base^(2i/dim))，i = 0,1,...,dim/2-1
    i = torch.arange(0, dim, 2, dtype=torch.float32, device=device)  # 步长为2，取偶数索引
    theta = 1.0 / (base ** (i / dim))   # 形状 (dim/2,)
    print("=== 频率计算 ===")
    print(f"i = {i}  (取偶数索引 0,2,4,...)")
    print(f"theta = {theta}  形状: {theta.shape}")

    # 位置索引
    positions = torch.arange(seq_len, dtype=torch.float32, device=device)  # (seq_len,)
    print(f"\npositions = {positions}  形状: {positions.shape}")

    # 外积：每个位置 × 每个频率 -> 角度矩阵 (seq_len, dim/2)
    angles = positions[:, None] * theta[None, :]   # (seq_len, dim/2)
    print(f"angles = positions * theta 形状: {angles.shape}")
    print("angles 值:\n", angles)

    # 计算 cos 和 sin，形状 (seq_len, dim/2)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    print(f"\ncos 形状: {cos.shape}, sin 形状: {sin.shape}")
    print("cos:\n", cos)
    print("sin:\n", sin)

    # 关键！扩展到 dim 维度：每个角度重复一次，但采用“前后拼接”方式
    # 例如 [c0, c1, c2] -> [c0, c1, c2, c0, c1, c2]
    cos = torch.cat([cos, cos], dim=-1)   # (seq_len, dim)
    sin = torch.cat([sin, sin], dim=-1)
    print(f"\n拼接后 cos 形状: {cos.shape}, sin 形状: {sin.shape}")
    print("拼接后 cos:\n", cos)
    return cos, sin


def rotate_half(x):
    """旋转辅助函数：把每对维度的第二个分量取负并与第一个交换"""
    # x 形状: (batch, n_heads, seq_len, dim) 或 (batch, seq_len, dim) ...
    x1 = x[..., :x.shape[-1] // 2]   # 前半段
    x2 = x[..., x.shape[-1] // 2:]   # 后半段
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(x, cos, sin):
    """
    将 RoPE 应用到输入 x 上
    x:      (batch, n_heads, seq_len, dim)
    cos, sin: (seq_len, dim)
    """
    # 维度对齐：cos/sin 需要扩展到和 x 一样的 batch 和 head 维度
    # (seq_len, dim) -> (1, 1, seq_len, dim)
    cos = cos.unsqueeze(0).unsqueeze(1)
    sin = sin.unsqueeze(0).unsqueeze(1)
    print(f"\n对齐后 cos 形状: {cos.shape}   (1,1,seq_len,dim)")
    print(f"x 形状: {x.shape}")

    rotated_x = rotate_half(x)
    print(f"rotate_half(x) 形状: {rotated_x.shape}")
    return (x * cos) + (rotated_x * sin)


# ---------- 测试小例子 ----------
print("===== RoPE 逐步演示 =====\n")
batch, n_heads, seq_len, dim = 1, 1, 3, 4   # 小维度方便观察
torch.manual_seed(42)
x = torch.randn(batch, n_heads, seq_len, dim)
print(f"输入 x 形状: {x.shape} = (batch, heads, seq_len, dim)")
print("x 值:")
print(x)

# 1. 预计算 cos, sin
cos, sin = precompute_rope_cos_sin(dim, seq_len, base=10000.0, device=x.device)

# 2. 应用旋转位置编码
print("\n===== 应用 RoPE =====")
output = apply_rotary_pos_emb(x, cos, sin)
print(f"\n输出 output 形状: {output.shape}")
print("输出 output 值:")
print(output)

===== RoPE 逐步演示 =====

输入 x 形状: torch.Size([1, 1, 3, 4]) = (batch, heads, seq_len, dim)
x 值:
tensor([[[[ 0.3367,  0.1288,  0.2345,  0.2303],
          [-1.1229, -0.1863,  2.2082, -0.6380],
          [ 0.4617,  0.2674,  0.5349,  0.8094]]]])
=== 频率计算 ===
i = tensor([0., 2.])  (取偶数索引 0,2,4,...)
theta = tensor([1.0000, 0.0100])  形状: torch.Size([2])

positions = tensor([0., 1., 2.])  形状: torch.Size([3])
angles = positions * theta 形状: torch.Size([3, 2])
angles 值:
 tensor([[0.0000, 0.0000],
        [1.0000, 0.0100],
        [2.0000, 0.0200]])

cos 形状: torch.Size([3, 2]), sin 形状: torch.Size([3, 2])
cos:
 tensor([[ 1.0000,  1.0000],
        [ 0.5403,  0.9999],
        [-0.4161,  0.9998]])
sin:
 tensor([[0.0000, 0.0000],
        [0.8415, 0.0100],
        [0.9093, 0.0200]])

拼接后 cos 形状: torch.Size([3, 4]), sin 形状: torch.Size([3, 4])
拼接后 cos:
 tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9999,  0.5403,  0.9999],
        [-0.4161,  0.9998, -0.4161,  0.9998]])

===== 应用 RoPE ==

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ---------- RoPE 工具函数（复用之前的设计）----------
def precompute_rope_cos_sin(dim, max_seq_len, base=10000.0, device='cpu'):
    """预计算 cos、sin 表，形状 (max_seq_len, dim)"""
    # dim 必须是偶数
    i = torch.arange(0, dim, 2, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (i / dim))          # (dim//2,)
    positions = torch.arange(max_seq_len, dtype=torch.float32, device=device)  # (max_seq_len,)
    print(positions.shape)
    angles = positions[:, None] * theta[None, :]  # (max_seq_len, dim//2)
    cos = torch.cos(angles)                     # (max_seq_len, dim//2)
    sin = torch.sin(angles)
    # 前后拼接，扩展到 dim 维
    cos = torch.cat([cos, cos], dim=-1)         # (max_seq_len, dim)
    sin = torch.cat([sin, sin], dim=-1)
    return cos, sin

def rotate_half(x):
    """旋转辅助：后半取负，拼到前半前面"""
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(x, cos, sin, seq_dim=2):
    """
    将 RoPE 应用到 x 上。
    x:      (batch, heads, seq_len, dim)
    cos, sin: (seq_len, dim) 或 (1, 1, seq_len, dim)
    """
    if cos.dim() == 2:
        # 维度对齐：(seq_len, dim) -> (1, 1, seq_len, dim)
        cos = cos[None, None, :, :]
        sin = sin[None, None, :, :]
    return (x * cos) + (rotate_half(x) * sin)


# ---------- 带 RoPE 的多头注意力 ----------
class MultiHeadAttentionWithRoPE(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, max_seq_len=128, base=10000.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能被 num_heads 整除"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.max_seq_len = max_seq_len

        # QKV 的线性投影（合并为一个矩阵，也可拆开）
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # 预计算 RoPE 的 cos 和 sin，缓存起来
        cos, sin = precompute_rope_cos_sin(self.head_dim, max_seq_len, base)
        self.register_buffer("cos", cos)  # (max_seq_len, head_dim)
        self.register_buffer("sin", sin)

    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, embed_dim)
        """
        batch, seq_len, _ = x.shape
        # 1. 线性投影得到 Q, K, V，并拆分成多头
        qkv = self.qkv(x)  # (batch, seq_len, 3*embed_dim)
        q, k, v = qkv.chunk(3, dim=-1)  # 每个都是 (batch, seq_len, embed_dim)

        # 重塑为 (batch, seq_len, num_heads, head_dim) -> 转置为 (batch, num_heads, seq_len, head_dim)
        q = q.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. 应用 RoPE 到 Q 和 K（只取前 seq_len 个位置的 cos/sin）
        cos = self.cos[:seq_len]  # (seq_len, head_dim)
        sin = self.sin[:seq_len]
        q = apply_rotary_pos_emb(q, cos, sin)
        k = apply_rotary_pos_emb(k, cos, sin)
        # v 不应用 RoPE

        # 3. 计算注意力分数
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (batch, heads, seq_len, seq_len)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)

        # 4. 加权求和
        out = torch.matmul(attn_weights, v)  # (batch, heads, seq_len, head_dim)

        # 5. 合并多头并投影
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, -1)  # (batch, seq_len, embed_dim)
        out = self.out_proj(out)
        return out


# ---------- 快速测试 ----------
if __name__ == "__main__":
    torch.manual_seed(42)
    batch, seq_len, embed_dim, heads = 2, 6, 8, 2
    x = torch.randn(batch, seq_len, embed_dim)
    mha = MultiHeadAttentionWithRoPE(embed_dim, heads, max_seq_len=seq_len)
    print(f"输入 x 形状: {x.shape}")
    print(f"cos 表形状: {mha.cos.shape}  (max_seq_len={mha.max_seq_len}, head_dim={heads})")
    out = mha(x)
    print(f"输出形状: {out.shape}")  # 应为 (2, 6, 8)

输入 x 形状: torch.Size([2, 6, 8])
cos 表形状: torch.Size([6, 4])  (max_seq_len=6, head_dim=2)
输出形状: torch.Size([2, 6, 8])


In [9]:
import torch, math

def apply_rope_simple(x, pos, base=10000.0):
    """x: 一维向量 (dim,), pos: 整数位置"""
    dim = x.shape[0]
    # 取偶数索引作为每个配对的基础频率
    i = torch.arange(0, dim, 2, dtype=torch.float32)
    theta = 1.0 / (base ** (i / dim))   # (dim//2,)
    # 每个配对对应的角度
    angles = pos * theta                # (dim//2,)
    cos = torch.cos(angles).repeat_interleave(2)  # (dim,)
    sin = torch.sin(angles).repeat_interleave(2)
    # 旋转辅助向量
    x_rot = torch.empty_like(x)
    x_rot[0::2] = -x[1::2]
    x_rot[1::2] =  x[0::2]
    return x * cos + x_rot * sin

# 原始向量
h = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("原始向量 h:", h)
for pos in [0, 1, 2, 5]:
    h_rope = apply_rope_simple(h, pos)
    print(f"位置 {pos}: {h_rope}")

原始向量 h: tensor([1., 2., 3., 4.])
位置 0: tensor([1., 2., 3., 4.])
位置 1: tensor([-1.1426,  1.9221,  2.9599,  4.0298])
位置 2: tensor([-2.2347,  0.0770,  2.9194,  4.0592])
位置 5: tensor([ 2.2015, -0.3916,  2.7963,  4.1449])


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def precompute_rope_cos_sin(dim, max_seq_len, base = 10000.0, device = 'cpu'):
    assert dim % 2 == 0, "dim must can // 2"
    i = torch.arange(0, dim, 2, dtype=torch.float32, device=device)
    positions = torch.arange(0, max_seq_len, 1, dtype=torch.float32, device=device)
    # positions = torch.arange(max_seq_len, dtype=torch.float32, device=device)
    print(positions.shape)
    theta = 1.0 / (base ** (i / dim))
    # angles = positions[:, None] * theta[None, :]
    angles = positions.unsqueeze(1) * theta.unsqueeze(0)
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    return cos, sin

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(x, cos, sin):
    if cos.dim == 2:
        cos = cos.unsqueeze(0).unsqueeze(1)
        sin = sin.unsqueeze(0).unsqueeze(1)
    return x * cos + rotate_half(x) * sin

# ---------- 带 RoPE 的多头注意力 ----------
class MultiHeadAttentionWithRoPE(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, max_seq_len=128, base=10000.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能被 num_heads 整除"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.max_seq_len = max_seq_len

        # QKV 的线性投影（合并为一个矩阵，也可拆开）
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # 预计算 RoPE 的 cos 和 sin，缓存起来
        cos, sin = precompute_rope_cos_sin(self.head_dim, max_seq_len, base)
        self.register_buffer("cos", cos)  # (max_seq_len, head_dim)
        self.register_buffer("sin", sin)

    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, embed_dim)
        """
        batch, seq_len, _ = x.shape
        # 1. 线性投影得到 Q, K, V，并拆分成多头
        qkv = self.qkv(x)  # (batch, seq_len, 3*embed_dim)
        q, k, v = qkv.chunk(3, dim=-1)  # 每个都是 (batch, seq_len, embed_dim)

        # 重塑为 (batch, seq_len, num_heads, head_dim) -> 转置为 (batch, num_heads, seq_len, head_dim)
        q = q.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. 应用 RoPE 到 Q 和 K（只取前 seq_len 个位置的 cos/sin）
        cos = self.cos[:seq_len]  # (seq_len, head_dim)
        sin = self.sin[:seq_len]
        q = apply_rotary_pos_emb(q, cos, sin)
        k = apply_rotary_pos_emb(k, cos, sin)
        # v 不应用 RoPE

        # 3. 计算注意力分数
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (batch, heads, seq_len, seq_len)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)

        # 4. 加权求和
        out = torch.matmul(attn_weights, v)  # (batch, heads, seq_len, head_dim)

        # 5. 合并多头并投影
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, -1)  # (batch, seq_len, embed_dim)
        out = self.out_proj(out)
        return out


# ---------- 快速测试 ----------
if __name__ == "__main__":
    torch.manual_seed(42)
    batch, seq_len, embed_dim, heads = 2, 6, 8, 2
    x = torch.randn(batch, seq_len, embed_dim)
    mha = MultiHeadAttentionWithRoPE(embed_dim, heads, max_seq_len=seq_len)
    print(f"输入 x 形状: {x.shape}")
    print(f"cos 表形状: {mha.cos.shape}  (max_seq_len={mha.max_seq_len}, head_dim={heads})")
    out = mha(x)
    print(f"输出形状: {out.shape}")  # 应为 (2, 6, 8)

torch.Size([6])
输入 x 形状: torch.Size([2, 6, 8])
cos 表形状: torch.Size([6, 4])  (max_seq_len=6, head_dim=2)
输出形状: torch.Size([2, 6, 8])


In [10]:
def precompute_rope_cos_sin(dim, max_seq_len, base=10000.0, device='cpu'):
    i = torch.arange(0, dim, 2, dtype=torch.float32, device=device)
    positions = torch.arange(0, max_seq_len, 1, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (i / dim))
    angles = positions[:, None] * theta[None, :]
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    return cos, sin

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(x, cos, sin):
    if cos.dim == 2:
        cos = cos[None, None, :, :]
        sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

# ---------- 带 RoPE 的多头注意力 ----------
class MultiHeadAttentionWithRoPE(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, max_seq_len=128, base=10000.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能被 num_heads 整除"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.max_seq_len = max_seq_len

        # QKV 的线性投影（合并为一个矩阵，也可拆开）
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # 预计算 RoPE 的 cos 和 sin，缓存起来
        cos, sin = precompute_rope_cos_sin(self.head_dim, max_seq_len, base)
        self.register_buffer("cos", cos)  # (max_seq_len, head_dim)
        self.register_buffer("sin", sin)

    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, embed_dim)
        """
        batch, seq_len, _ = x.shape
        # 1. 线性投影得到 Q, K, V，并拆分成多头
        qkv = self.qkv(x)  # (batch, seq_len, 3*embed_dim)
        q, k, v = qkv.chunk(3, dim=-1)  # 每个都是 (batch, seq_len, embed_dim)

        # 重塑为 (batch, seq_len, num_heads, head_dim) -> 转置为 (batch, num_heads, seq_len, head_dim)
        q = q.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. 应用 RoPE 到 Q 和 K（只取前 seq_len 个位置的 cos/sin）
        cos = self.cos[:seq_len]  # (seq_len, head_dim)
        sin = self.sin[:seq_len]
        q = apply_rotary_pos_emb(q, cos, sin)
        k = apply_rotary_pos_emb(k, cos, sin)
        # v 不应用 RoPE

        # 3. 计算注意力分数
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (batch, heads, seq_len, seq_len)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)

        # 4. 加权求和
        out = torch.matmul(attn_weights, v)  # (batch, heads, seq_len, head_dim)

        # 5. 合并多头并投影
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, -1)  # (batch, seq_len, embed_dim)
        out = self.out_proj(out)
        return out


# ---------- 快速测试 ----------
if __name__ == "__main__":
    torch.manual_seed(42)
    batch, seq_len, embed_dim, heads = 2, 6, 8, 2
    x = torch.randn(batch, seq_len, embed_dim)
    mha = MultiHeadAttentionWithRoPE(embed_dim, heads, max_seq_len=seq_len)
    print(f"输入 x 形状: {x.shape}")
    print(f"cos 表形状: {mha.cos.shape}  (max_seq_len={mha.max_seq_len}, head_dim={heads})")
    out = mha(x)
    print(f"输出形状: {out.shape}")  # 应为 (2, 6, 8)

输入 x 形状: torch.Size([2, 6, 8])
cos 表形状: torch.Size([6, 4])  (max_seq_len=6, head_dim=2)
输出形状: torch.Size([2, 6, 8])


In [11]:
import torch, math

# 假设向量 x = [0.5, 1.0, -0.3, 0.8, 0.2, -0.6, 1.2, -0.4]
x = torch.tensor([0.5, 1.0, -0.3, 0.8, 0.2, -0.6, 1.2, -0.4])
dim = 8
pos = 1
base = 10000

# 频率 (dim/2 = 4 个)
i = torch.arange(0, dim, 2).float()
theta = 1.0 / (base ** (i / dim))
# theta ≈ [1.0, 0.1, 0.01, 0.001] （简化示意）
# 实际计算略

# 相邻配对
cos_adj = torch.cos(pos * theta).repeat_interleave(2)
sin_adj = torch.sin(pos * theta).repeat_interleave(2)
x_rot_adj = torch.empty_like(x)
x_rot_adj[0::2] = -x[1::2]
x_rot_adj[1::2] =  x[0::2]
result_adj = x * cos_adj + x_rot_adj * sin_adj

# 首尾配对
cos_cat = torch.cat([torch.cos(pos * theta), torch.cos(pos * theta)])
sin_cat = torch.cat([torch.sin(pos * theta), torch.sin(pos * theta)])
def rotate_half(x):
    return torch.cat((-x[dim//2:], x[:dim//2]))
result_cat = x * cos_cat + rotate_half(x) * sin_cat

print("相邻配对结果:", result_adj)
print("首尾配对结果:", result_cat)
print("两者相等?", torch.allclose(result_adj, result_cat))

相邻配对结果: tensor([-0.5713,  0.9610, -0.3784,  0.7661,  0.2060, -0.5980,  1.2004, -0.3988])
首尾配对结果: tensor([ 0.1019,  1.0549, -0.3120,  0.8004,  0.5288, -0.4972,  1.1969, -0.3992])
两者相等? False
